# Notebook 1 — Exploratory Data Analysis

**Goal:** Understand the Telco churn dataset before any modelling.

We'll look at:
- Target class distribution (imbalance check)
- Churn rate by key categorical features
- Numeric feature distributions
- Correlation heatmap

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_data, summarize

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.2)
plt.rcParams['figure.dpi'] = 120

In [ ]:
# ── Load data ──────────────────────────────────────────────────────────────────
df = load_data('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
summarize(df)
df.head()

## 1. Target Distribution

First thing: check how imbalanced the classes are.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
counts = df['Churn'].value_counts()
axes[0].bar(['No Churn', 'Churn'], counts.values, color=['#4C72B0', '#DD8452'], width=0.5)
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontsize=11)

# Pie chart
axes[1].pie(counts.values, labels=['No Churn', 'Churn'],
            autopct='%1.1f%%', colors=['#4C72B0', '#DD8452'],
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Churn Rate')

plt.suptitle('Target Variable — Churn', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f'\nChurn rate: {df["Churn"].mean():.1%}')
print('→ Class imbalance present — we will use SMOTE during training.')

## 2. Churn Rate by Key Categorical Features

In [ ]:
cat_cols = ['Contract', 'PaymentMethod', 'InternetService', 'tenure']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col in zip(axes, ['Contract', 'PaymentMethod', 'InternetService']):
    churn_rate = df.groupby(col)['Churn'].mean().sort_values(ascending=False)
    bars = ax.bar(churn_rate.index, churn_rate.values * 100,
                  color='#4C72B0', edgecolor='white')
    ax.set_title(f'Churn Rate by {col}', fontsize=12)
    ax.set_ylabel('Churn Rate (%)')
    ax.set_ylim(0, 55)
    ax.tick_params(axis='x', rotation=20)
    ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)

plt.tight_layout()
plt.show()

print('Key insight: Month-to-month customers churn at 3-4x the rate of two-year contracts.')

## 3. Numeric Feature Distributions by Churn

In [ ]:
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col in zip(axes, numeric_cols):
    for churn_val, label, color in [(0, 'No Churn', '#4C72B0'), (1, 'Churn', '#DD8452')]:
        subset = df[df['Churn'] == churn_val][col]
        ax.hist(subset, bins=30, alpha=0.6, label=label, color=color, density=True)
    ax.set_title(col)
    ax.set_xlabel(col)
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)

plt.suptitle('Numeric Feature Distributions — Churned vs Not Churned',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print('Key insight:')
print('  - Low tenure → higher churn (new customers leave early)')
print('  - Higher monthly charges → slightly more churn')

## 4. Correlation Heatmap

In [ ]:
corr = df[['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']].corr()

fig, ax = plt.subplots(figsize=(6, 5))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True,
            linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix — Numeric Features', pad=12)
plt.tight_layout()
plt.show()

## EDA Summary

| Finding | Implication |
|---------|-------------|
| ~26% churn rate | Class imbalance — use F1, not accuracy |
| Month-to-month contract = 43% churn | Strong predictor |
| Low tenure = higher churn | Tenure buckets will help |
| High monthly charges = slight churn increase | charges_ratio feature |

→ **Next:** Feature engineering in Notebook 02